In [ ]:
# Initialization
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='geeexercise') # my project name

In [ ]:
# Configurations
aoi = ee.FeatureCollection('projects/geeexercise/assets/gebe-island-aoi-land').geometry()
aoi_area_ha = round(aoi.area(1).getInfo() / 1e4, 2)
print(f"AOI loaded: {aoi_area_ha} ha")

years = range(2019, 2026)
crs, scale = 'EPSG:32752', 10
export_folder = 'GEE_Exports'

# Sensor params
s2_col = 'COPERNICUS/S2_SR_HARMONIZED'
s1_col = 'COPERNICUS/S1_GRD'
cs_thr = 0.65
s1_orbit = 'DESCENDING'  # default, verified below

# Quick check
print("Running quick data check...")
asc_cnt, desc_cnt = 0, 0
s2_flags = []

for yr in years:
    dates = f'{yr}-01-01', f'{yr}-12-31'

    # S1 orbit tally
    s1 = ee.ImageCollection(s1_col).filterBounds(aoi).filterDate(*dates)
    asc_cnt += s1.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')).size().getInfo()
    desc_cnt += s1.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')).size().getInfo()

    # S2 usability check
    s2 = ee.ImageCollection(s2_col).filterBounds(aoi).filterDate(*dates)
    usable = s2.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)).size().getInfo()
    s2_flags.append(usable)
    print(f"  {yr}: S1(A={asc_cnt}, D={desc_cnt}) - S2 usable: {usable}")

s1_orbit = 'DESCENDING' if desc_cnt >= asc_cnt else 'ASCENDING'
print(f"\nDominant orbit: {s1_orbit}")
if any(c < 5 for c in s2_flags):
    print("Some years have <5 raw S2 scenes.")

In [ ]:
# Preprocessing functions
def mask_cs2(img):
    """Apply CS+ mask. Pixels >= threshold are kept."""
    cs = img.select('cs_cdf')
    return img.updateMask(cs.gte(cs_thr))

def get_s2_stack(yr):
    """Build CS+ linked S2 collection for a year."""
    dates = f'{yr}-01-01', f'{yr}-12-31'
    s2 = ee.ImageCollection(s2_col).filterBounds(aoi).filterDate(*dates)
    csp = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED') \
             .filterBounds(aoi).filterDate(*dates).select('cs_cdf')
    return s2.linkCollection(csp, ['cs_cdf']).map(mask_cs2)

def valid_px_count(col):
    """Count unmasked observations per pixel across the image collections."""
    return col.select('B4').map(lambda i: i.mask().rename('vcount')).sum().toInt16().clip(aoi)

def lee_filter(img, ksize=3):
    """Adaptive Lee filter in linear scale, returns dB."""
    lin = ee.Image(10).pow(img.divide(10.0))
    kernel = ee.Kernel.square(radius=ksize//2, units='pixels', normalize=False)

    loc_mean = lin.reduceNeighborhood(ee.Reducer.mean(), kernel)
    loc_var  = lin.reduceNeighborhood(ee.Reducer.variance(), kernel)

    # Noise variance approximation
    noise = loc_var.reduceRegion(ee.Reducer.mean(), geometry=img.geometry(), scale=100, maxPixels=1e8)
    noise_img = ee.Image.constant(noise.values(loc_var.bandNames())).rename(loc_var.bandNames())

    k = loc_var.divide(loc_var.add(noise_img)).clamp(0, 1)
    filt_lin = loc_mean.add(k.multiply(lin.subtract(loc_mean)))
    return filt_lin.log10().multiply(10.0).rename(img.bandNames())

def build_composites(yr):
    """Return S2 composite, S2 valid count, and S1 composite for a given year."""
    dates = f'{yr}-01-01', f'{yr}-12-31'

    # S2
    s2_col_masked = get_s2_stack(yr)
    s2_comp = s2_col_masked.select(['B2','B3','B4','B8','B11','B12']).median().clip(aoi).set('year', yr)
    vcount = valid_px_count(s2_col_masked)

    # S1
    s1_raw = ee.ImageCollection(s1_col).filterBounds(aoi).filterDate(*dates) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('orbitProperties_pass', s1_orbit)).select(['VV','VH'])

    s1_comp = s1_raw.map(lambda i: lee_filter(i, 3)).median().clip(aoi).set('year', yr)

    return s2_comp, vcount, s1_comp

In [ ]:
# Execution and export
tasks = []
print("Building graphs and submitting exports ...")

for yr in years:
    s2, vc, s1 = build_composites(yr)

    # S2 composite
    tasks.append(ee.batch.Export.image.toDrive(
        s2.toFloat(), f'S2_Composite_Gebe_{yr}', export_folder,
        region=aoi, scale=scale, crs=crs, maxPixels=1e10))

    # S1 composite
    tasks.append(ee.batch.Export.image.toDrive(
        s1.toFloat(), f'S1_Composite_Gebe_{yr}', export_folder,
        region=aoi, scale=scale, crs=crs, maxPixels=1e10))

# Fire them off
for t in tasks: t.start()
print(f"\nSubmitted {len(tasks)} tasks. Monitor at: https://code.earthengine.google.com/tasks")

In [ ]:
# QA Visualization
# Store as lazy objects to avoid rebuilding the graph when rendering
s2_map, s1_map, vc_map = {}, {}, {}
for yr in years:
    s2_map[yr], vc_map[yr], s1_map[yr] = build_composites(yr)

Map = geemap.Map(center=[0.083, 129.467], zoom=11)
Map.add_basemap('SATELLITE')
Map.addLayer(aoi, {'color': 'white'}, 'AOI')

# Toggle layers for QA
for yr in years:
    Map.addLayer(s2_map[yr], {'min':0, 'max':3000, 'bands':['B4','B3','B2']}, f'S2 True Colour {yr}', shown=False)
    Map.addLayer(s1_map[yr].select('VV'), {'min':-25, 'max':0}, f'S1 VV {yr}', shown=False)
    Map.addLayer(vc_map[yr], {'min':0, 'max':20, 'palette':['#d73027','#fee08b','#1a9850']}, f'Valid Pixels {yr}', shown=(yr==2025))

Map.add_colorbar({'min':0, 'max':20, 'palette':['#d73027','#fee08b','#1a9850']}, label='Valid Observations (0=red, 20+=green)')
Map.addLayerControl()
Map